# Multi-Regional AeroMAPS - Execution Time Scaling Benchmark

This notebook benchmarks how the execution time of `MultiRegionalProcess` grows with the number of regions.

It reads `regionalisation_all_regions_supertest.yaml` (which contains many region copies pointing to the same underlying configs), then subsets it to increasing values of `N` and times the full `compute()` call for each `N`.


## Setup

In [ ]:
import gc
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import yaml

from aeromaps import create_multi_regional_process

%matplotlib widget
warnings.filterwarnings("ignore")

BASE_YAML = Path("regionalisation_all_regions_supertest.yaml")
TMP_YAML = Path("_tmp_scaling_run.yaml")

## Helpers

Load the base yaml once, then write temporary yaml files that expose only the first `N` regions.

In [ ]:
with open(BASE_YAML) as f:
    base_cfg = yaml.safe_load(f)

all_region_items = list(base_cfg["regionalisation"]["regions"].items())
MAX_REGIONS = len(all_region_items)
print(f"Available regions in supertest yaml: {MAX_REGIONS}")


def write_subset_yaml(n_regions: int, path: Path = TMP_YAML) -> Path:
    """Write a yaml containing only the first `n_regions` regions from the base config."""
    cfg = yaml.safe_load(yaml.safe_dump(base_cfg))  # deep copy
    cfg["regionalisation"]["regions"] = dict(all_region_items[:n_regions])
    with open(path, "w") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    return path


def time_compute(n_regions: int) -> dict:
    """Create a process with `n_regions` regions and time both init and compute."""
    write_subset_yaml(n_regions)

    gc.collect()
    t0 = time.perf_counter()
    process = create_multi_regional_process(
        configuration_file=str(TMP_YAML), disable_execution_statistics=True
    )
    t1 = time.perf_counter()
    process.compute(parallel=True)
    t2 = time.perf_counter()

    return {
        "n_regions": n_regions,
        "init_s": t1 - t0,
        "compute_s": t2 - t1,
        "total_s": t2 - t0,
    }

In [ ]:
region_counts = [1, 2, 5, 10, 20, 40, 60, 80, 100]
region_counts = [n for n in region_counts if n <= MAX_REGIONS]

results = []
for n in region_counts:
    print(f"→ Running with N={n} region(s)...", flush=True)
    r = time_compute(n)
    results.append(r)
    print(f"   init={r['init_s']:.2f}s  compute={r['compute_s']:.2f}s  total={r['total_s']:.2f}s")

print("\nDone.")

## Results table and scaling plot

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
df["init_per_region_s"] = df["init_s"] / df["n_regions"]
df["compute_per_region_s"] = df["compute_s"] / df["n_regions"]
df["total_per_region_s"] = df["total_s"] / df["n_regions"]
df

In [ ]:
ns = df["n_regions"].to_numpy()

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

ax = axes[0]
ax.plot(ns, df["init_s"], "o-", label="init")
ax.plot(ns, df["compute_s"], "s-", label="compute")
ax.plot(ns, df["total_s"], "^-", label="total", linewidth=2)
# linear reference fitted on the total
if len(ns) >= 2:
    slope = df["total_s"].iloc[-1] / ns[-1]
    ax.plot(ns, slope * ns, "k--", alpha=0.4, label="linear ref (total)")
ax.set_xlabel("Number of regions")
ax.set_ylabel("Time [s]")
ax.set_title("Execution time vs number of regions")
ax.grid(True, alpha=0.3)
ax.legend()

ax = axes[1]
ax.plot(ns, df["init_per_region_s"], "o-", label="init / region")
ax.plot(ns, df["compute_per_region_s"], "s-", label="compute / region")
ax.plot(ns, df["total_per_region_s"], "^-", label="total / region", linewidth=2)
ax.set_xlabel("Number of regions")
ax.set_ylabel("Time per region [s]")
ax.set_title("Amortised cost per region")
ax.grid(True, alpha=0.3)
ax.legend()

fig.tight_layout()
plt.show()

In [ ]:
# Cleanup the temporary yaml written during the loop
if TMP_YAML.exists():
    TMP_YAML.unlink()